In [1]:
!nvidia-smi
# This is a shell command (the "!" tells Colab to run it in the terminal, not Python).
# nvidia-smi prints your GPU model, driver version, and current memory usage —
# confirms you actually have a T4 attached before you waste time debugging later.

Sun Jul 12 17:45:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers faiss-cpu datasets gradio
# -q            : quiet install, less scroll noise
# transformers  : loads the LLM + tokenizer
# accelerate    : helps transformers place model layers on GPU efficiently
# bitsandbytes  : enables 4-bit quantization (shrinks the model to fit T4's 16GB VRAM)
# sentence-transformers : the embedding model that turns text into vectors
# faiss-cpu     : vector similarity search engine (cpu version is fine for this dataset size)
# datasets      : Hugging Face's dataset loader, used to pull MedQuAD
# gradio        : builds the chat web UI at the end

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 74.8 MB/s eta 0:00:00


In [3]:
from datasets import load_dataset
# Imports Hugging Face's dataset-loading function.

dataset = load_dataset("lavita/MedQuAD")
# Downloads MedQuAD directly from the Hugging Face Hub and caches it in this Colab session.
# This is a CC BY 4.0 licensed dataset of ~47,000 medical Q&A pairs sourced from 12
# official NIH websites (MedlinePlus, cancer.gov, NIDDK, etc.) — exactly built for this use case.

df = dataset["train"].to_pandas()
# Converts the Hugging Face dataset object into a pandas DataFrame — easier to inspect/edit.

print(df.shape)
# Prints (num_rows, num_columns) — sanity check that something actually loaded.

print(df.columns.tolist())
# IMPORTANT: prints the exact column names. Different re-uploads of MedQuAD sometimes
# name columns slightly differently (e.g. "answer" vs "Answer" vs "response").
# Check this output BEFORE running Step 3 — if the names differ, adjust the
# df["answer"] / row['question'] references in Step 3 to match what you see here.

print(df.head(3))
# Previews the first 3 rows so you can visually confirm question/answer/source content looks right.

README.md:   0%|          | 0.00/2.77k [00:00<?, ?B/s]

data/train-00000-of-00001-e36383d177026d(…):   0%|          | 0.00/10.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/47441 [00:00<?, ? examples/s]

(47441, 13)
['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms', 'question_id', 'question_focus', 'question_type', 'question', 'answer']
  document_id document_source  \
0     0000559             GHR   
1     0000559             GHR   
2     0000559             GHR   

                                        document_url category  umls_cui  \
0  https://ghr.nlm.nih.gov/condition/keratoderma-...     None  C0343073   
1  https://ghr.nlm.nih.gov/condition/keratoderma-...     None  C0343073   
2  https://ghr.nlm.nih.gov/condition/keratoderma-...     None  C0343073   

  umls_semantic_types umls_semantic_group synonyms question_id  \
0                T047           Disorders     KWWH   0000559-1   
1                T047           Disorders     KWWH   0000559-2   
2                T047           Disorders     KWWH   0000559-3   

                 question_focus    question_type  \
0  keratoderma with woolly hair 

In [36]:
import re

def clean_url(url):
    match = re.search(r"https?://[^\s\)\]]+", str(url))
    # Finds just the actual URL pattern inside the messy string and ignores everything else —
    # the markdown brackets/parens/extra text around it get discarded.
    return match.group(0) if match else url

documents = [
    {
        "text": f"Q: {row['question']}\nA: {row['answer']}",
        "source": row.get("document_source", "MedQuAD"),
        "url": clean_url(row.get("document_url", ""))
    }
    for _, row in df.iterrows()
]

In [37]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")
# Loads a small, fast embedding model onto the GPU.
# all-MiniLM-L6-v2 is a great beginner default: 80MB, fast, good semantic quality.

texts = [doc["text"] for doc in documents]
# Extracts just the text strings to feed into the embedder.

embeddings = embedder.encode(texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True)
# Converts every text chunk into a fixed-length vector (384 dimensions for this model).
# batch_size=64 uses the T4 efficiently without running out of memory.

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/257 [00:00<?, ?it/s]

In [38]:
import faiss
import numpy as np

dimension = embeddings.shape[1]
# Gets the vector size (384) so FAISS knows what shape of index to build.

index = faiss.IndexFlatL2(dimension)
# Creates a simple, exact-search FAISS index using Euclidean (L2) distance.
# "Flat" means it checks every vector — fine for tens of thousands of docs.

index.add(embeddings.astype("float32"))
# Loads all your document embeddings into the index so they can be searched.

faiss.write_index(index, "medirag.index")
# Saves the index to disk so you don't have to re-embed every time you restart Colab.

In [39]:
def retrieve(query, k=3):
    query_vec = embedder.encode([query], convert_to_numpy=True).astype("float32")
    # Embeds the user's question the same way we embedded the documents —
    # they must live in the same vector space to be compared.

    distances, indices = index.search(query_vec, k)
    # Searches the FAISS index for the k closest document vectors to the query vector.

    results = [documents[i] for i in indices[0]]
    # Maps the returned indices back to the actual text + source.

    return results

In [40]:
docs = retrieve("what are the symptoms of diabetes", k=3)
for d in docs:
    print(d["source"], "-", d["text"][:150])
    # Prints the first 150 characters of each retrieved chunk.
    # If these chunks look unrelated to your question, the problem is in
    # embedding/indexing (Steps 4-5), not the LLM.

NIHSeniorHealth - Q: What are the symptoms of Diabetes ?
A: Many people with diabetes experience one or more symptoms, including extreme thirst or hunger, a frequent ne
NIHSeniorHealth - Q: What are the symptoms of Diabetes ?
A: Diabetes is often called a "silent" disease because it can cause serious complications even before you have 
NIDDK - Q: What are the symptoms of Your Guide to Diabetes: Type 1 and Type 2 ?
A: The signs and symptoms of diabetes are - being very thirsty - urinating oft


In [21]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "microsoft/Phi-3-mini-4k-instruct"
# The generator model — small enough for a T4, strong enough for grounded Q&A.

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
# Configures 4-bit quantization: shrinks memory footprint ~4x with minor quality loss —
# this is what makes an otherwise-too-big model fit on a free T4.

tokenizer = AutoTokenizer.from_pretrained(model_id)
# Loads the tokenizer that converts text <-> token IDs the model understands.

model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=bnb_config, device_map="auto"
)
# Loads the model weights in 4-bit precision and automatically places them on the GPU.

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

In [22]:
def build_prompt(query, retrieved_docs):
    context = "\n\n".join([f"[Source: {d['source']}]\n{d['text']}" for d in retrieved_docs])
    # Joins the retrieved chunks into one context block, each labeled with its source
    # so the model (and the user) can see where each fact came from.

    prompt = f"""You are a careful medical information assistant. Answer the user's question
using ONLY the context provided below. If the context doesn't contain the answer, say you
don't have enough information. Always remind the user this is not a diagnosis and to consult
a healthcare professional for serious or persistent symptoms. Cite the source after each fact.

Context:
{context}

Question: {query}

Answer:"""
    # This instruction block is called "grounding" — it forces the model to rely on retrieved
    # text instead of its own (possibly wrong) internal medical knowledge, and bakes in the
    # safety disclaimer directly into the generation.
    return prompt

In [25]:
def answer_question(query):
    docs = retrieve(query, k=3)
    prompt = build_prompt(query, docs)

    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to("cuda")
    # return_dict=True forces a consistent output: a dict with "input_ids" and
    # "attention_mask" tensors inside — this avoids the ambiguous behavior that
    # caused the KeyError, since we're now always indexing explicitly rather than
    # assuming what type apply_chat_template hands back.

    input_ids = inputs["input_ids"]
    # Pulls the actual token-ID tensor out of the dict.

    output = model.generate(
        **inputs, max_new_tokens=300, temperature=0.3, do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    # **inputs unpacks both input_ids and attention_mask as keyword args to generate().

    response = tokenizer.decode(output[0][input_ids.shape[-1]:], skip_special_tokens=True)
    # Now input_ids is guaranteed to be a real tensor, so .shape works correctly.

    return response, docs

In [26]:
answer, docs = answer_question("what are the symptoms of diabetes")
print(answer)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


The symptoms of diabetes include:

1. Being very thirsty
2. Frequent urination
3. Feeling very hungry
4. Feeling very tired
5. Losing weight without trying
6. Sores that heal slowly
7. Dry, itchy skin
8. Loss of feeling or tingling in the feet
9. Blurry eyesight

It's important to note that some people with diabetes may not experience any of these symptoms. The only way to know if you have diabetes is to have your doctor perform a blood test.


In [27]:
for d in docs:
    print(d["source"])

MedQuAD
MedQuAD
MedQuAD


In [28]:
print(df.columns.tolist())

['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms', 'question_id', 'question_focus', 'question_type', 'question', 'answer']


In [11]:
EMERGENCY_KEYWORDS = ["chest pain", "can't breathe", "suicidal", "severe bleeding", "unconscious"]
# A simple keyword list to catch clearly urgent situations before even calling the LLM.

def safety_check(query):
    lowered = query.lower()
    if any(word in lowered for word in EMERGENCY_KEYWORDS):
        return "⚠️ This sounds like it could be a medical emergency. Please call your local emergency number or go to the nearest ER immediately. I'm not able to help with urgent situations."
    return None
    # If this returns a string, the app shows it instead of calling the LLM at all —
    # a real safety layer, not just a prompt-level disclaimer.

In [41]:
import gradio as gr

def chat(query, history):
    emergency = safety_check(query)
    if emergency:
        return emergency
    # Checks for emergencies first — cheapest and most important check.

    answer, docs = answer_question(query)
    # Runs the full RAG pipeline.

    sources = "\n".join(set(f"{d['source']} ({d['url']})" for d in docs))
    # Builds "SourceName (url)" pairs for each unique source used —
    # lets the user click through to verify the answer instead of just seeing a name.

    return f"{answer}\n\n📚 Sources:\n{sources}"

demo = gr.ChatInterface(
    fn=chat,
    title="MediRAG — Evidence-Grounded Symptom Checker",
    description="Answers are grounded in NIH/MedlinePlus data. Not a substitute for professional medical advice."
)
demo.launch(share=True)
# share=True gives you a public link straight from Colab — great for demo GIFs and testing on mobile.

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://88cc3f3fbe6daabc34.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [42]:
!git config --global user.email "rajibchatterjee1205@gmail.com"
!git config --global user.name "RAJIB-VERSE"
# Replace both with your actual email and GitHub username.

In [43]:
!git clone https://github.com/YOUR-USERNAME/medirag.git
%cd medirag
# Replace YOUR-USERNAME with your actual GitHub username.
# This pulls your empty repo into Colab's filesystem and moves into that folder.

Cloning into 'medirag'...
fatal: could not read Username for 'https://github.com': No such device or address
[Errno 2] No such file or directory: 'medirag'
/content


In [44]:
!mkdir -p notebooks assets rag data

In [ ]:
from google.colab import files
uploaded = files.upload()
# Opens a file picker in Colab — select README.md from your computer (the one I generated for you).